In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500])
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import os, pickle, json, glob, zipfile, warnings
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_MAP= {'Low': 0, 'Medium': 1, 'High': 2}
INV_LABEL= {0: 'Low', 1: 'Medium', 2: 'High'}
WINDOW_SIZE= 30
N_FEATURES= 8

In [ ]:
TRAIN_PKL= '/kaggle/input/datasets/risha8750/phase5-outputs/train_samples.pkl'
VAL_PKL= '/kaggle/input/datasets/risha8750/acl-phase5-pkl/val_samples.pkl'
WEIGHTS_JSON = '/kaggle/input/datasets/risha8750/acl-phase5-pkl/class_weights.json'
PHASE3_CSV= '/kaggle/input/datasets/risha8750/acl-phase3-extracted/phase3_keypoints_master.csv'
COLLEGE_CSV= '/kaggle/input/datasets/ziya07/college-sports-real-time-skill-feedback-dataset/college_sports_training_dataset.csv'
SP_ROOT= '/kaggle/input/datasets/risha8750/sportspose-poses/data'
OUT_DIR= '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
with open(TRAIN_PKL, 'rb') as f:
    train_samples = pickle.load(f)

with open(VAL_PKL, 'rb') as f:
    val_samples = pickle.load(f)

with open(WEIGHTS_JSON, 'r') as f:
    raw_weights = json.load(f)

print(f"Train samples : {len(train_samples)}")
print(f"Val samples   : {len(val_samples)}")
print(f"Class weights : {raw_weights}")

sample = train_samples[0]
assert sample['features'].shape == (30, 8), f"Unexpected feature shape: {sample['features'].shape}"
print("Feature shape confirmed: (30, 8)")
print("Keys in sample:", list(sample.keys()))

In [ ]:
from collections import Counter
import numpy as np

def label_dist(samples):
    return dict(Counter(s['label'] for s in samples))

def source_dist(samples):
    return dict(Counter(s['source'] for s in samples))

train_dist   = label_dist(train_samples)
train_sources = source_dist(train_samples)

print("Train label distribution:", train_dist)
print("Train sources:", train_sources)
print("Val label distribution:", label_dist(val_samples))

total = sum(train_dist.values())
for cls, cnt in train_dist.items():
    print(f"  {cls:<10}: {cnt:>8,}  ({100*cnt/total:.2f}%)")

In [ ]:
import random
random.seed(42)

low_samples= [s for s in train_samples if s['label'] == 'Low']
medium_samples = [s for s in train_samples if s['label'] == 'Medium']
high_samples= [s for s in train_samples if s['label'] == 'High']

print(f"Before rebalance — Low: {len(low_samples)}, Medium: {len(medium_samples)}, High: {len(high_samples)}")

n_high   = len(high_samples)
n_medium = len(medium_samples)
cap_low  = max(4 * n_high, 4 * n_medium, 5000)
cap_med  = max(2 * n_high, n_medium)

low_samples    = random.sample(low_samples, min(cap_low, len(low_samples)))
medium_samples = random.sample(medium_samples, min(cap_med, len(medium_samples)))

train_samples = low_samples + medium_samples + high_samples
random.shuffle(train_samples)

print(f"After rebalance  — Low: {len(low_samples)}, Medium: {len(medium_samples)}, High: {len(high_samples)}")
print(f"Total training samples: {len(train_samples)}")
print("New distribution:", label_dist(train_samples))

In [ ]:
new_dist = label_dist(train_samples)
total    = sum(new_dist.values())
inv_freq = {k: total / (3.0 * v) for k, v in new_dist.items()}
norm_sum = sum(inv_freq.values())
updated_weights = {k: round(v / norm_sum, 4) for k, v in inv_freq.items()}

print("Recomputed class weights:", updated_weights)

class_weights_tensor = torch.tensor(
    [updated_weights['Low'], updated_weights['Medium'], updated_weights['High']],
    dtype=torch.float32
).to(DEVICE)

In [ ]:
import numpy as np

sample_features = [s['features'] for s in train_samples[:500]]
arr = np.array(sample_features)

print("Feature statistics across 500 training samples:")
feature_names = ['l_knee_angle', 'r_knee_angle', 'l_hip_angle', 'r_hip_angle',
                 'trunk_lean', 'asymmetry', 'l_knee_vel', 'r_knee_vel']

for i, name in enumerate(feature_names):
    col = arr[:, :, i].flatten()
    print(f"  {name:<18}  min={col.min():.3f}  max={col.max():.3f}  "
          f"mean={col.mean():.3f}  median={np.median(col):.3f}")

print()
sub_keys = ['S_knee', 'S_asym', 'S_trunk', 'S_velocity']
for k in sub_keys:
    vals = [s['sub_scores'][k] for s in train_samples[:500] if k in s.get('sub_scores', {})]
    if vals:
        print(f"  {k:<15}  mean={np.mean(vals):.4f}  median={np.median(vals):.4f}  "
              f"min={np.min(vals):.4f}  max={np.max(vals):.4f}")

print()
score_vals = [s['score'] for s in train_samples[:500]]
print(f"  Composite score  mean={np.mean(score_vals):.4f}  "
      f"median={np.median(score_vals):.4f}  "
      f"min={np.min(score_vals):.4f}  max={np.max(score_vals):.4f}")

In [ ]:
def linear_sub_score(value, safe_max, risk_min):
    if value <= safe_max:
        return 0.0
    if value >= risk_min:
        return 1.0
    return float((value - safe_max) / (risk_min - safe_max))

def compute_sub_scores_from_window(win):
    l_knee = win[:, 0]
    r_knee = win[:, 1]
    trunk  = win[:, 4]
    asym   = win[:, 5]
    vel    = (np.abs(win[:, 6]) + np.abs(win[:, 7])) / 2.0

    KNEE_NEUTRAL   = 170.0
    KNEE_SAFE_MIN  = 140.0
    KNEE_RISK_MIN  = 100.0

    knee_avg = (np.nanmean(l_knee) + np.nanmean(r_knee)) / 2.0
    s_knee = 0.0
    if knee_avg < KNEE_NEUTRAL:
        deviation = KNEE_NEUTRAL - knee_avg
        max_dev   = KNEE_NEUTRAL - KNEE_RISK_MIN
        safe_dev  = KNEE_NEUTRAL - KNEE_SAFE_MIN
        if deviation <= safe_dev:
            s_knee = 0.0
        elif deviation >= max_dev:
            s_knee = 1.0
        else:
            s_knee = float((deviation - safe_dev) / (max_dev - safe_dev))

    TRUNK_NEUTRAL  = 180.0
    TRUNK_SAFE_MIN = 170.0
    TRUNK_RISK_MIN = 155.0

    trunk_avg = np.nanmean(np.abs(trunk))
    trunk_dev = TRUNK_NEUTRAL - trunk_avg
    safe_trunk_dev = TRUNK_NEUTRAL - TRUNK_SAFE_MIN
    risk_trunk_dev = TRUNK_NEUTRAL - TRUNK_RISK_MIN
    if trunk_dev <= safe_trunk_dev:
        s_trunk = 0.0
    elif trunk_dev >= risk_trunk_dev:
        s_trunk = 1.0
    else:
        s_trunk = float((trunk_dev - safe_trunk_dev) / (risk_trunk_dev - safe_trunk_dev))

    s_asym     = linear_sub_score(float(np.nanmean(np.abs(asym))), 5.0, 15.0)
    s_velocity = linear_sub_score(float(np.nanpercentile(np.abs(vel), 90)), 150.0, 300.0)

    return s_knee, s_asym, s_trunk, s_velocity

def assign_composite_label(win):
    s_knee, s_asym, s_trunk, s_vel = compute_sub_scores_from_window(win)
    score = 0.40 * s_knee + 0.30 * s_asym + 0.15 * s_trunk + 0.15 * s_vel
    if score <= 0.33:
        label = 'Low'
    elif score <= 0.66:
        label = 'Medium'
    else:
        label = 'High'
    return score, label, {'S_knee': s_knee, 'S_asym': s_asym, 'S_trunk': s_trunk, 'S_velocity': s_vel}

In [ ]:
def relabel_samples(samples):
    relabeled = []
    for s in samples:
        s_copy = dict(s)
        score, label, sub_scores = assign_composite_label(s['features'])
        s_copy['score']      = score
        s_copy['label']      = label
        s_copy['sub_scores'] = sub_scores
        relabeled.append(s_copy)
    return relabeled

print("Relabeling train set...")
train_samples = relabel_samples(train_samples)
print("Relabeling val set...")
val_samples   = relabel_samples(val_samples)

print("New train distribution:", label_dist(train_samples))
print("New val distribution  :", label_dist(val_samples))

low_pct  = label_dist(train_samples).get('Low', 0) / len(train_samples)
high_pct = label_dist(train_samples).get('High', 0) / len(train_samples)
print(f"\nLow: {low_pct:.2%}   High: {high_pct:.2%}")

In [ ]:
import random
random.seed(42)

low_s  = [s for s in train_samples if s['label'] == 'Low']
med_s  = [s for s in train_samples if s['label'] == 'Medium']
high_s = [s for s in train_samples if s['label'] == 'High']

n_high  = len(high_s)
cap_med = int(n_high * 1.5)
cap_low = n_high

med_s  = random.sample(med_s, min(cap_med, len(med_s)))
low_s  = random.sample(low_s, min(cap_low, len(low_s)))

train_samples = low_s + med_s + high_s
random.shuffle(train_samples)

print("Rebalanced train distribution:", label_dist(train_samples))
print(f"Total training samples       : {len(train_samples):,}")

In [ ]:
def label_dist(samples):
    return dict(Counter(s['label'] for s in samples))

def source_dist(samples):
    return dict(Counter(s['source'] for s in samples))

print("Train labels :", label_dist(train_samples))
print("Val labels   :", label_dist(val_samples))
print("Train sources:", source_dist(train_samples))

high_pct = label_dist(train_samples).get('High', 0) / len(train_samples)
print(f"\nHigh Risk fraction in train: {high_pct:.2%}")
if high_pct < 0.15:
    print("WARNING: High Risk below 15% — class weighting is critical.")

In [ ]:
def make_windows(features, source, view, seq_key, stride=15):
    T = features.shape[0]
    samples = []
    i = 0
    while i + WINDOW_SIZE <= T:
        win = features[i:i + WINDOW_SIZE].astype(np.float32)
        if not np.isfinite(win).all():
            win = np.nan_to_num(win, nan=0.0, posinf=0.0, neginf=0.0)
        score, label, sub_scores = assign_composite_label(win)
        samples.append({
            'features'  : win,
            'label'     : label,
            'score'     : score,
            'sub_scores': sub_scores,
            'source'    : source,
            'view'      : view,
            'seq_key'   : f"{seq_key}-{i}"
        })
        i += stride
    return samples

def resolve_col(df, candidates):
    low_cols = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns:
            return c
        if c.lower() in low_cols:
            return low_cols[c.lower()]
    return None

In [ ]:
df3 = pd.read_csv(PHASE3_CSV)
print("Phase3 CSV shape:", df3.shape)
print("Columns:", df3.columns.tolist()[:20])

seq_col = None
for c in ['seq_id', 'sequence_id', 'clip_id', 'video_id', 'file', 'subject']:
    if c in df3.columns:
        seq_col = c
        break
if seq_col is None:
    seq_col = df3.columns[0]
    print(f"Using first column as seq_col: {seq_col}")

feat_map = {
    0: ['l_knee_angle', 'left_knee_angle', 'knee_angle_left', 'knee_l'],
    1: ['r_knee_angle', 'right_knee_angle', 'knee_angle_right', 'knee_r'],
    2: ['l_hip_angle',  'left_hip_angle',  'hip_angle_left',  'hip_l'],
    3: ['r_hip_angle',  'right_hip_angle', 'hip_angle_right', 'hip_r'],
    4: ['trunk_lean',   'trunk_angle',     'torso_lean'],
    5: ['asymmetry',    'l_r_asym',        'knee_asym'],
    6: ['l_knee_vel',   'left_knee_vel',   'knee_vel_l'],
    7: ['r_knee_vel',   'right_knee_vel',  'knee_vel_r']
}
resolved3 = {i: resolve_col(df3, cands) for i, cands in feat_map.items()}
print("Resolved Phase3 columns:", resolved3)

phase3_supp = []
for seq_key, grp in df3.groupby(seq_col):
    arr = np.zeros((len(grp), 8), dtype=np.float32)
    for col_idx, col_name in resolved3.items():
        if col_name:
            arr[:, col_idx] = pd.to_numeric(grp[col_name], errors='coerce').fillna(0).values
    if arr.shape[0] < WINDOW_SIZE:
        continue
    if resolved3.get(5) is None:
        arr[:, 5] = np.abs(arr[:, 0] - arr[:, 1])
    if resolved3.get(6) is None:
        arr[:, 6] = np.gradient(arr[:, 0]) * 30.0
    if resolved3.get(7) is None:
        arr[:, 7] = np.gradient(arr[:, 1]) * 30.0
    phase3_supp += make_windows(arr, 'Phase3_MediaPipe', 'front', str(seq_key))

print(f"Phase3 supplemental windows: {len(phase3_supp)}")

In [ ]:
df_col = pd.read_csv(COLLEGE_CSV)
print("College CSV shape:", df_col.shape)
print("Columns:", df_col.columns.tolist())

acl_acts = {'jump', 'sprint', 'kick', 'lunge', 'Jump', 'Sprint', 'Kick', 'Lunge',
             'JUMP', 'SPRINT', 'KICK', 'LUNGE'}
act_col = resolve_col(df_col, ['Activity', 'activity', 'action', 'Action', 'movement_type'])
if act_col:
    df_col = df_col[df_col[act_col].isin(acl_acts)].reset_index(drop=True)
    print(f"Filtered to ACL-relevant activities: {len(df_col)} rows")

sess_col = resolve_col(df_col, ['session_id', 'Session_ID', 'subject_id', 'Subject',
                                 'athlete_id', 'Athlete', 'trial_id'])
if sess_col is None:
    df_col['_seq'] = df_col.index // (WINDOW_SIZE * 3)
    sess_col = '_seq'

college_map = {
    0: ['Knee_L_Angle', 'knee_angle_left', 'Left_Knee_Angle',  'KneeAngle_L', 'knee_l'],
    1: ['Knee_R_Angle', 'knee_angle_right','Right_Knee_Angle', 'KneeAngle_R', 'knee_r'],
    2: ['Hip_L_Angle',  'hip_angle_left',  'Left_Hip_Angle',   'HipAngle_L',  'hip_l'],
    3: ['Hip_R_Angle',  'hip_angle_right', 'Right_Hip_Angle',  'HipAngle_R',  'hip_r'],
    4: ['Trunk_Lean',   'trunk_angle',     'TrunkAngle',       'trunk'],
    5: ['Asymmetry',    'l_r_diff',        'KneeAsymmetry',    'asym'],
    6: ['Knee_L_Vel',   'Left_Knee_Velocity', 'KneeVel_L'],
    7: ['Knee_R_Vel',   'Right_Knee_Velocity','KneeVel_R']
}
col_resolved = {i: resolve_col(df_col, cands) for i, cands in college_map.items()}

college_supp = []
for seq_key, grp in df_col.groupby(sess_col):
    arr = np.zeros((len(grp), 8), dtype=np.float32)
    for col_idx, col_name in col_resolved.items():
        if col_name:
            arr[:, col_idx] = pd.to_numeric(grp[col_name], errors='coerce').fillna(0).values
    if arr.shape[0] < WINDOW_SIZE:
        continue
    if col_resolved.get(5) is None:
        arr[:, 5] = np.abs(arr[:, 0] - arr[:, 1])
    if col_resolved.get(6) is None:
        arr[:, 6] = np.gradient(arr[:, 0]) * 30.0
    if col_resolved.get(7) is None:
        arr[:, 7] = np.gradient(arr[:, 1]) * 30.0
    college_supp += make_windows(arr, 'College_Sports', 'front', str(seq_key))

print(f"College Sports supplemental windows: {len(college_supp)}")

In [ ]:
def angle_3pts(a, b, c):
    v1 = a - b
    v2 = c - b
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-9 or n2 < 1e-9:
        return 0.0
    cos_val = np.dot(v1, v2) / (n1 * n2)
    return float(np.degrees(np.arccos(np.clip(cos_val, -1.0, 1.0))))

def trunk_lean_3d(shoulder, hip):
    vec = shoulder - hip
    up = np.array([0.0, 1.0, 0.0])
    n = np.linalg.norm(vec)
    if n < 1e-9:
        return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(vec / n, up), -1.0, 1.0))))

def build_features_from_poses(poses):
    T = poses.shape[0]
    l_k_raw = np.array([angle_3pts(poses[t,11], poses[t,13], poses[t,15]) for t in range(T)])
    r_k_raw = np.array([angle_3pts(poses[t,12], poses[t,14], poses[t,16]) for t in range(T)])
    l_h_raw = np.array([angle_3pts(poses[t, 5], poses[t,11], poses[t,13]) for t in range(T)])
    r_h_raw = np.array([angle_3pts(poses[t, 6], poses[t,12], poses[t,14]) for t in range(T)])
    tr_raw  = np.array([trunk_lean_3d(
        (poses[t,5] + poses[t,6]) / 2,
        (poses[t,11] + poses[t,12]) / 2
    ) for t in range(T)])

    wl = min(7, T) if min(7, T) % 2 != 0 else min(6, T - 1)
    if T >= wl and wl >= 3:
        l_knee = savgol_filter(l_k_raw, wl, 2)
        r_knee = savgol_filter(r_k_raw, wl, 2)
    else:
        l_knee, r_knee = l_k_raw, r_k_raw

    asym  = np.abs(l_knee - r_knee)
    l_vel = np.gradient(l_knee) * 30.0
    r_vel = np.gradient(r_knee) * 30.0

    return np.stack([l_knee, r_knee, l_h_raw, r_h_raw, tr_raw, asym, l_vel, r_vel], axis=1)

In [ ]:
RELEVANT_ACTS = {'jump', 'soccer'}
sportspose_supp = []

for env in ['indoors', 'outdoors']:
    env_path = os.path.join(SP_ROOT, env)
    if not os.path.isdir(env_path):
        print(f"Skipping missing: {env_path}")
        continue
    for subject in sorted(os.listdir(env_path)):
        subj_path = os.path.join(env_path, subject)
        if not os.path.isdir(subj_path):
            continue
        for activity in sorted(os.listdir(subj_path)):
            if activity not in RELEVANT_ACTS:
                continue
            act_path = os.path.join(subj_path, activity)
            if not os.path.isdir(act_path):
                continue
            for npy_file in sorted(glob.glob(os.path.join(act_path, '*.npy'))):
                try:
                    raw = np.load(npy_file)
                    if raw.ndim == 3 and raw.shape[1] == 17 and raw.shape[2] == 3:
                        poses = raw
                    elif raw.ndim == 2 and raw.shape[1] == 51:
                        poses = raw.reshape(-1, 17, 3)
                    elif raw.ndim == 2 and raw.shape[1] == 17 * 3:
                        poses = raw.reshape(-1, 17, 3)
                    else:
                        continue
                    if poses.shape[0] < WINDOW_SIZE:
                        continue
                    features = build_features_from_poses(poses)
                    seq_key  = f"{env}_{subject}_{activity}_{os.path.splitext(os.path.basename(npy_file))[0]}"
                    sportspose_supp += make_windows(features, 'SportsPose', 'front', seq_key)
                except Exception as e:
                    continue

print(f"SportsPose supplemental windows: {len(sportspose_supp)}")

In [ ]:
running_supp = []
print("Running Biomechanics dataset skipped — Kaggle network access unavailable.")
print(f"Current train samples: {len(train_samples):,}")

In [ ]:
all_supp = phase3_supp + college_supp + sportspose_supp + running_supp
print(f"Total supplemental windows: {len(all_supp)}")

if len(all_supp) > 0:
    print("Supplemental label dist:", label_dist(all_supp))
    train_samples = train_samples + all_supp

final_dist = label_dist(train_samples)
print("Final training distribution:", final_dist)

total = sum(final_dist.values())
inv_freq = {k: total / (3.0 * v) for k, v in final_dist.items()}
norm_sum = sum(inv_freq.values())
updated_weights = {k: v / norm_sum for k, v in inv_freq.items()}
print("Updated class weights:", {k: round(v, 4) for k, v in updated_weights.items()})

class_weights_tensor = torch.tensor(
    [updated_weights['Low'], updated_weights['Medium'], updated_weights['High']],
    dtype=torch.float32
).to(DEVICE)

with open(os.path.join(OUT_DIR, 'class_weights_final.json'), 'w') as f:
    json.dump(updated_weights, f, indent=2)

In [ ]:
class RiskSequenceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        x = torch.tensor(s['features'], dtype=torch.float32)
        y = torch.tensor(LABEL_MAP[s['label']], dtype=torch.long)
        return x, y

In [ ]:
train_dataset = RiskSequenceDataset(train_samples)
val_dataset   = RiskSequenceDataset(val_samples)

train_int_labels = [LABEL_MAP[s['label']] for s in train_samples]
class_counts     = np.bincount(train_int_labels, minlength=3)
sample_wts       = [1.0 / (class_counts[l] + 1e-9) for l in train_int_labels]

sampler = WeightedRandomSampler(
    weights=sample_wts,
    num_samples=len(sample_wts),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}  ({len(train_dataset)} samples)")
print(f"Val batches   : {len(val_loader)}  ({len(val_dataset)} samples)")

In [ ]:
class ACLRiskLSTM(nn.Module):
    def __init__(self, input_size=8, hidden_size=128, num_layers=2,
                 num_classes=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_scores = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = (lstm_out * attn_weights).sum(dim=1)
        return self.classifier(context)


model = ACLRiskLSTM().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_dec ay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

EPOCHS   = 80
PATIENCE = 12

best_val_f1      = 0.0
no_improve_count = 0
best_model_path  = os.path.join(OUT_DIR, 'best_acl_lstm.pt')

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss = 0.0
    tr_preds, tr_true = [], []

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        run_loss += loss.item() * xb.size(0)
        tr_preds.extend(logits.argmax(1).cpu().tolist())
        tr_true.extend(yb.cpu().tolist())

    train_loss = run_loss / len(train_dataset)
    train_f1   = f1_score(tr_true, tr_preds, average='macro', zero_division=0)

    model.eval()
    v_loss = 0.0
    v_preds, v_true = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            v_loss += criterion(logits, yb).item() * xb.size(0)
            v_preds.extend(logits.argmax(1).cpu().tolist())
            v_true.extend(yb.cpu().tolist())

    val_loss = v_loss / len(val_dataset)
    val_f1   = f1_score(v_true, v_preds, average='macro', zero_division=0)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)

    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve_count = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"Epoch {epoch:3d}  |  Train Loss {train_loss:.4f}  Val Loss {val_loss:.4f}  "
              f"|  Train F1 {train_f1:.4f}  Val F1 {val_f1:.4f}  [saved]")
    else:
        no_improve_count += 1
        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d}  |  Train Loss {train_loss:.4f}  Val Loss {val_loss:.4f}  "
                  f"|  Train F1 {train_f1:.4f}  Val F1 {val_f1:.4f}")

    if no_improve_count >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}")
        break

print(f"\nBest validation F1: {best_val_f1:.4f}")

In [ ]:
epochs_ran = len(history['train_loss'])
ep_range   = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ep_range, history['train_loss'], label='Train Loss', color='steelblue', lw=2)
ax1.plot(ep_range, history['val_loss'],   label='Val Loss',   color='tomato',    lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ep_range, history['train_f1'], label='Train F1 Macro', color='steelblue', lw=2)
ax2.plot(ep_range, history['val_f1'],   label='Val F1 Macro',   color='tomato',    lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score (Macro)')
ax2.set_title('Training and Validation F1 Macro')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_probs, all_preds, all_true = [], [], []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_true.extend(yb.tolist())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_true  = np.array(all_true)

y_bin = label_binarize(all_true, classes=[0, 1, 2])
auroc_by_class = {}
for i, cls in enumerate(['Low', 'Medium', 'High']):
    if y_bin[:, i].sum() > 0:
        auroc_by_class[cls] = float(roc_auc_score(y_bin[:, i], all_probs[:, i]))

macro_auroc = float(np.mean(list(auroc_by_class.values())))
f1_macro    = float(f1_score(all_true, all_preds, average='macro', zero_division=0))
f1_per_cls  = f1_score(all_true, all_preds, average=None, zero_division=0)

cm         = confusion_matrix(all_true, all_preds, labels=[0, 1, 2])
tp_high    = cm[2, 2]
fn_high    = cm[2, 0] + cm[2, 1]
fnr_high   = float(fn_high / (tp_high + fn_high + 1e-9))

In [ ]:
print("=" * 55)
print("EVALUATION RESULTS")
print("=" * 55)
print(f"Macro AUROC          : {macro_auroc:.4f}")
print(f"Macro F1             : {f1_macro:.4f}")
print(f"High Risk FNR        : {fnr_high:.4f}  ({fn_high} missed out of {tp_high + fn_high})")
print()
print("Per-class AUROC:")
for cls, val in auroc_by_class.items():
    print(f"  {cls:<10}: {val:.4f}")
print()
print("Per-class F1:")
for i, cls in enumerate(['Low', 'Medium', 'High']):
    print(f"  {cls:<10}: {f1_per_cls[i]:.4f}")
print()
print(classification_report(
    all_true, all_preds,
    target_names=['Low', 'Medium', 'High'],
    zero_division=0
))

results_dict = {
    'macro_auroc'   : macro_auroc,
    'macro_f1'      : f1_macro,
    'fnr_high_risk' : fnr_high,
    'auroc_by_class': auroc_by_class,
    'f1_by_class'   : {c: float(f1_per_cls[i]) for i, c in enumerate(['Low', 'Medium', 'High'])}
}
with open(os.path.join(OUT_DIR, 'eval_results.json'), 'w') as f:
    json.dump(results_dict, f, indent=2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Low', 'Medium', 'High'],
    yticklabels=['Low', 'Medium', 'High'],
    ax=ax, linewidths=0.5
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — Validation Set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: confusion_matrix.png")

In [ ]:
colors = {'Low': 'steelblue', 'Medium': 'darkorange', 'High': 'tomato'}

fig, ax = plt.subplots(figsize=(8, 6))
for i, cls in enumerate(['Low', 'Medium', 'High']):
    if y_bin[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    auc_val      = auroc_by_class.get(cls, 0.0)
    ax.plot(fpr, tpr, label=f"{cls} (AUC = {auc_val:.3f})",
            color=colors[cls], lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('One-vs-Rest ROC Curves — Validation Set', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: roc_curves.png")

In [ ]:
val_true_np  = np.array([LABEL_MAP[s['label']] for s in val_samples])
val_pred_np  = all_preds
val_src_arr  = np.array([s.get('source', 'unknown') for s in val_samples])

high_mask   = val_true_np == 2
fn_mask     = high_mask & (val_pred_np != 2)
tp_mask     = high_mask & (val_pred_np == 2)

def mean_sub_scores(indices):
    keys = ['S_knee', 'S_asym', 'S_trunk', 'S_velocity']
    result = {k: [] for k in keys}
    for idx in indices:
        ss = val_samples[idx].get('sub_scores', {})
        for k in keys:
            result[k].append(ss.get(k, 0.0))
    return {k: float(np.mean(v)) if v else 0.0 for k, v in result.items()}

fn_indices = np.where(fn_mask)[0]
tp_indices = np.where(tp_mask)[0]

fn_avg = mean_sub_scores(fn_indices)
tp_avg = mean_sub_scores(tp_indices)

print(f"High Risk windows  : {int(high_mask.sum())}")
print(f"  Correctly caught : {len(tp_indices)}  (TP)")
print(f"  Missed           : {len(fn_indices)}  (FN)")
print()
print("Mean sub-scores — Missed High Risk (FN):")
for k, v in fn_avg.items():
    print(f"  {k:<15}: {v:.4f}")
print()
print("Mean sub-scores — Caught High Risk (TP):")
for k, v in tp_avg.items():
    print(f"  {k:<15}: {v:.4f}")

sub_df = pd.DataFrame({'FN_mean': fn_avg, 'TP_mean': tp_avg})
sub_df.to_csv(os.path.join(OUT_DIR, 'fn_subscore_analysis.csv'))
print("\nSaved: fn_subscore_analysis.csv")

In [ ]:
sources_in_val = list(set(s.get('source', 'unknown') for s in val_samples))
print("Sources in val set:", sources_in_val)

rows = []
for src in sorted(sources_in_val):
    src_indices = [i for i, s in enumerate(val_samples) if s.get('source') == src]
    if len(src_indices) < 5:
        continue
    src_true = val_true_np[src_indices]
    src_pred = val_pred_np[src_indices]
    src_probs = all_probs[src_indices]
    src_bin  = label_binarize(src_true, classes=[0, 1, 2])

    src_f1 = float(f1_score(src_true, src_pred, average='macro', zero_division=0))
    src_auroc = 0.0
    valid_cls = [i for i in range(3) if src_bin[:, i].sum() > 0]
    if len(valid_cls) >= 2:
        src_auroc = float(roc_auc_score(
            src_bin[:, valid_cls], src_probs[:, valid_cls],
            average='macro', multi_class='ovr'
        ))
    rows.append({'source': src, 'n_samples': len(src_indices),
                 'macro_f1': src_f1, 'macro_auroc': src_auroc})

breakdown_df = pd.DataFrame(rows)
print(breakdown_df.to_string(index=False))
breakdown_df.to_csv(os.path.join(OUT_DIR, 'cross_source_breakdown.csv'), index=False)
print("\nSaved: cross_source_breakdown.csv")

In [ ]:
cmu_val = [s for s in val_samples if s.get('source') == 'CMU']
if not cmu_val:
    cmu_val = val_samples

demo_seq_key = cmu_val[0].get('seq_key', 'demo')
base_key     = demo_seq_key.rsplit('-', 1)[0]

seq_windows  = [s for s in val_samples if s.get('seq_key', '').startswith(base_key)]
seq_windows.sort(key=lambda s: int(s.get('seq_key', '-0').rsplit('-', 1)[-1]))

if len(seq_windows) < 2:
    seq_windows = cmu_val[:10]

model.eval()
risk_scores = []
with torch.no_grad():
    for s in seq_windows:
        x     = torch.tensor(s['features'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
        prob  = torch.softmax(model(x), dim=1).cpu().numpy()[0]
        risk_scores.append(prob[2])

knee_angles = [float(s['features'][:, 0].mean()) for s in seq_windows]
true_labels = [s['label'] for s in seq_windows]
color_map   = {'Low': 'green', 'Medium': 'orange', 'High': 'red'}
bar_colors  = [color_map.get(l, 'gray') for l in true_labels]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.bar(range(len(risk_scores)), risk_scores, color=bar_colors, alpha=0.8, width=0.8)
ax1.axhline(0.67, color='red',    linestyle='--', lw=1.5, label='High Risk threshold (0.67)')
ax1.axhline(0.34, color='orange', linestyle='--', lw=1.5, label='Medium Risk threshold (0.34)')
ax1.set_ylabel('High Risk Probability', fontsize=11)
ax1.set_title(f'Frame-by-Frame Risk Score Timeline  —  Sequence: {base_key}', fontsize=12)
ax1.set_ylim(0, 1)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(range(len(knee_angles)), knee_angles, color='steelblue', lw=2, marker='o', ms=4)
ax2.set_xlabel('Window Index (each = 30 frames)', fontsize=11)
ax2.set_ylabel('Mean Left Knee Angle (°)', fontsize=11)
ax2.set_title('Left Knee Angle Across Windows', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'risk_timeline.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: risk_timeline.png  ({len(seq_windows)} windows shown)")

In [ ]:
torch.save({
    'epoch'           : EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state' : optimizer.state_dict(),
    'best_val_f1'     : best_val_f1,
    'macro_auroc'     : macro_auroc,
    'fnr_high_risk'   : fnr_high,
    'label_map'       : LABEL_MAP,
    'feature_cols'    : ['l_knee_angle', 'r_knee_angle', 'l_hip_angle', 'r_hip_angle',
                         'trunk_lean', 'asymmetry', 'l_knee_vel', 'r_knee_vel'],
    'class_weights'   : updated_weights,
    'window_size'     : WINDOW_SIZE,
    'architecture'    : 'BiLSTM_AttentionPool_2layer_hidden128',
    'training_sources': ['CMU_reprojection', 'Penn_proxy', 'Phase3_MediaPipe',
                         'College_Sports', 'SportsPose', 'Running_Biomechanics'],
    'known_limitations': [
        '2D angle estimates carry up to 18 degree error vs 3D clinical systems',
        'Penn proxy labels derived from formula, not verified injury outcomes',
        'No confirmed injury outcome data in any training set',
        'Camera angle must be front-facing or sagittal within plus or minus 10 degrees'
    ]
}, os.path.join(OUT_DIR, 'acl_risk_model_phase6.pt'))

print("Model saved: acl_risk_model_phase6.pt")

In [ ]:
outputs = [
    'best_acl_lstm.pt',
    'acl_risk_model_phase6.pt',
    'class_weights_final.json',
    'eval_results.json',
    'training_curves.png',
    'confusion_matrix.png',
    'roc_curves.png',
    'risk_timeline.png',
    'fn_subscore_analysis.csv',
    'cross_source_breakdown.csv'
]

print("=" * 45)
print("PHASE 6 OUTPUT FILES")
print("=" * 45)
for fname in outputs:
    fpath = os.path.join(OUT_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:<40} {size_kb:>8.1f} KB")
    else:
        print(f"  {fname:<40}   NOT FOUND")

print()
print(f"Best Val F1 (macro)  : {best_val_f1:.4f}")
print(f"Macro AUROC          : {macro_auroc:.4f}")
print(f"High Risk FNR        : {fnr_high:.4f}")
print()
print("Phase 6 complete.")